# Lab 28: Incident Response for a Deployed AI Detector

*AI Cybersecurity Masterclass · Lab 28 · AI60secLab*

**Companion video:** [▶ How AI Security Incident Response Works in 60 Seconds](https://www.youtube.com/watch?v=c19cfYykn6w&list=PLG4l6Ssi9A00)

**Research question:** How do you know an AI control failed after the system went live, and how do you recover?

A detector runs for 30 simulated days on the UNSW-NB15 test set, replayed in daily batches. Two incidents happen on purpose: a poisoned model update that blinds the detector to one attack category, and a telemetry change that rescales byte counters. Operators only see monitoring signals and a small analyst-reviewed sample.

**Learning objectives**
- build a baseline before you need it;
- monitor decisions and trust boundaries, not every detail;
- treat a signal as the start of an investigation, not proof;
- triage into scope and severity, contain first, preserve evidence;
- recover to a known-good state and measure that it worked.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
import matplotlib.pyplot as plt

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import random_forest_pipeline
from src.ai_monitoring import (
    ModelRegistry,
    daily_batches,
    fingerprint,
    monitor_day,
    baseline_limits,
    signals,
    scale_bytes,
)

train, test = load_unsw(DATA_DIR)
train_sample = train.sample(60000, random_state=0).reset_index(drop=True)

X_train, y_train = split_xy(train_sample, "label")
X_test, y_test = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)
X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)


def fit(X, y):
    model = random_forest_pipeline(X)
    model.set_params(model__n_estimators=100)
    model.fit(X, y)
    return model


registry = ModelRegistry()
v1 = fit(X_train, y_train)
registry.register("v1", v1, fingerprint(y_train.tolist()), "trusted training data")

# Incident 1 (hidden from operators): an update trained on tampered labels.
WATCH = "Fuzzers"
poisoned_labels = y_train.copy()
poisoned_labels[train_sample["attack_cat"] == WATCH] = 0
v2 = fit(X_train, poisoned_labels)
registry.register("v2", v2, fingerprint(poisoned_labels.tolist()), "routine retrain (labels tampered upstream)")

print("Training-data fingerprints:", {k: v["training_data"] for k, v in registry.versions.items()})

## Step 1: The deployment, with a response playbook

The loop below is what operations sees each day. The response rules are written in advance:
- score drift right after a **model change**: roll back to the last known-good version;
- score drift with **no model change**: suspect the input pipeline; pause automated actions and fix the pipeline;
- an alert-rate wobble with no drift: open a ticket and watch, do not act.

Containment takes effect the day after detection, to model triage time.

In [ ]:
batches = daily_batches(X_test, y_test, test["attack_cat"], days=30, seed=0)
reference_scores = v1.predict_proba(batches[0][0])[:, 1]

BASELINE_DAYS = range(1, 10)
PSI_INVESTIGATE = 0.10  # common rule of thumb: PSI above 0.1 is a meaningful shift

deployed, pipeline_broken, pending = "v1", False, None
last_model_change = -99
rows, actions = [], []

for day, (X_day, y_day, cat_day) in enumerate(batches):
    if day == 10:
        deployed, last_model_change = "v2", day
        registry.deploy("v2", day, "scheduled retrain")
    if day == 20:
        pipeline_broken = True  # incident 2: byte counters change units upstream

    if pending:
        kind = pending
        if kind == "rollback":
            deployed = "v1"
            registry.deploy("v1", day, "rollback after drift following a model change")
        if kind == "fix_pipeline":
            pipeline_broken = False
        actions.append({"day": day, "action": kind})
        pending = None

    model = registry.versions[deployed]["model"]
    X_in = scale_bytes(X_day) if pipeline_broken else X_day
    observed = monitor_day(model, X_in, y_day, cat_day, reference_scores, watch_category=WATCH, seed=day)
    observed.update({"day": day, "model": deployed, "pipeline_broken": pipeline_broken})
    rows.append(observed)

    if day >= 10 and observed["score_psi"] > PSI_INVESTIGATE:
        evidence = {"day": day, "model": deployed, "psi": round(observed["score_psi"], 3)}
        if day - last_model_change <= 2 and deployed != "v1":
            pending = "rollback"
        elif deployed == "v1":
            pending = "fix_pipeline"
        actions.append({"day": day, "action": f"signal -> investigate {evidence}"})

timeline = pd.DataFrame(rows).set_index("day")
display(pd.DataFrame(actions))

## Step 2: What the operators saw

In [ ]:
limits = baseline_limits(timeline.loc[list(BASELINE_DAYS)], ["alert_rate", "score_psi", "feedback_recall", "feedback_precision"])
flagged = signals(timeline, limits)
visible = ["model", "alert_rate", "score_psi", "feedback_recall", "feedback_precision", "any_signal"]
display(flagged[visible].round(3))

Two lessons hide in this table:
- an **alert-rate band** built from the baseline can fire on a normal day. A signal starts an investigation; it does not prove an attack;
- during the poisoned-model days, alert volume **fell** and precision **rose**. A dashboard tracking only those would call the poisoned model an improvement.

## Step 3: The ground truth operators could not see

In [ ]:
truth_columns = ["model", "true_recall", "true_precision", f"true_{WATCH.lower()}_recall"]
display(timeline[truth_columns].round(3))

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(timeline.index, timeline["score_psi"], marker="o")
axes[0].axhline(PSI_INVESTIGATE, linestyle="--")
axes[0].set_ylabel("score PSI (visible)")
axes[0].set_title("Monitoring signal versus hidden impact")
axes[1].plot(timeline.index, timeline[f"true_{WATCH.lower()}_recall"], marker="o", label=f"{WATCH} recall (hidden)")
axes[1].plot(timeline.index, timeline["true_precision"], marker="o", label="precision (hidden)")
axes[1].set_ylabel("ground truth")
axes[1].set_xlabel("day")
axes[1].legend()
plt.tight_layout()
plt.show()

## Step 4: Evidence and recovery

Preserve what you need to explain the incident later: which version ran when, and the fingerprint of the data each was trained on. Then measure that recovery returned behavior to baseline.

In [ ]:
display(pd.DataFrame(registry.history))
display(pd.DataFrame({k: {"training_data": v["training_data"], "note": v["note"]} for k, v in registry.versions.items()}).T)

phases = {
    "baseline (days 1-9)": range(1, 10),
    "poisoned model live": [d for d in timeline.index if timeline.loc[d, "model"] == "v2"],
    "pipeline broken": [d for d in timeline.index if timeline.loc[d, "pipeline_broken"]],
    "after recovery (days 25-29)": range(25, 30),
}
recovery = pd.DataFrame({
    name: timeline.loc[list(days), ["true_recall", "true_precision", f"true_{WATCH.lower()}_recall", "alert_rate"]].mean()
    for name, days in phases.items()
}).T
display(recovery.round(3))

## Interpretation

1. Which signal caught each incident, and how many days was the system exposed?
2. Why did the playbook choose rollback for one incident and a pipeline fix for the other?
3. The poisoned model *improved* precision. Which monitoring would have caught the blind spot directly?
4. What would you add to the signals, the controls, the tests, and the response process after this incident?

**Takeaway:** you need telemetry and a baseline before you can see change. Contain first, preserve evidence, recover to a trusted state, and let every incident improve the next response.